# Imports

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Flatten, BatchNormalization, Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dropout, Rescaling
import pathlib
import keras
import keras_tuner

import tensorflow as tf

## Download dataset

In [3]:

DATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATASET_NAME = "flower_photos"
CACHE_DIR = "."

data_dir = pathlib.Path(
    tf.keras.utils.get_file(
        origin    = DATASET_URL,
        fname     = DATASET_NAME,   # nome del file .tgz salvato
        cache_dir = CACHE_DIR,
        untar     = True,              # estrae automaticamente
        extract   = True,
    )
)

### Parameters

In [4]:
IMG_HEIGHT = 180
IMG_WIDTH  = 180
BATCH_SIZE = 32
SEED       = 123

VAL_SPLIT  = 0.2
AUTOTUNE = tf.data.AUTOTUNE

### Util functions

In [5]:
def configure_for_performance(ds):
  ds = ds.cache()
  ds = ds.shuffle(buffer_size=1000)
  ds = ds.batch(BATCH_SIZE)
  ds = ds.prefetch(buffer_size=AUTOTUNE)
  return ds

In [6]:
train_ds = keras.utils.image_dataset_from_directory(
  data_dir / DATASET_NAME,
  validation_split=VAL_SPLIT,
  subset="training",
  seed=SEED,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE)

val_ds = keras.utils.image_dataset_from_directory(
  data_dir / DATASET_NAME,
  validation_split=VAL_SPLIT,
  subset="validation",
  seed=SEED,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE)

# train_ds = configure_for_performance(train_ds)
# val_ds = configure_for_performance(val_ds)

Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.


### Layers

In [7]:
def build_model(hp):
    model = Sequential()
    model.add(tf.keras.layers.Rescaling(1./255))
    model.add(Conv2D(hp.Choice('units_1', [8, 16, 32]), hp.Choice('kernel_1', [3, 5, 7]), activation=hp.Choice('act_1', ['relu', 'leaky_relu'])))
    model.add(MaxPooling2D())
    model.add(Conv2D(hp.Choice('units_2', [8, 16, 32]), hp.Choice('kernel_2', [3, 5, 7]), activation=hp.Choice('act_2', ['relu', 'leaky_relu'])))
    model.add(MaxPooling2D())
    model.add(Conv2D(hp.Choice('units_3', [8, 16, 32]), hp.Choice('kernel_3', [3, 5, 7]), activation=hp.Choice('act_3', ['relu', 'leaky_relu'])))
    model.add(MaxPooling2D())
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(5))

    model.compile(
        optimizer='adam',
        loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'])

    return model

In [ ]:
# Altra opzione senza random search
model = Sequential()
model.add(Rescaling(1./255))

model.add(Conv2D(32, 5, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(64, 3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(128, 3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(256, 3, activation='relu'))
model.add(BatchNormalization())
model.add(GlobalAveragePooling2D())

model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(5))

model.compile(
    optimizer='adamw',
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [18]:
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 46s 464ms/step - accuracy: 0.4595 - loss: 1.3595 - val_accuracy: 0.2480 - val_loss: 1.7726
Epoch 2/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 42s 451ms/step - accuracy: 0.5375 - loss: 1.1466 - val_accuracy: 0.2466 - val_loss: 1.8287
Epoch 3/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 45s 490ms/step - accuracy: 0.6029 - loss: 1.0295 - val_accuracy: 0.2888 - val_loss: 1.7963
Epoch 4/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 44s 481ms/step - accuracy: 0.6349 - loss: 0.9798 - val_accuracy: 0.2984 - val_loss: 1.8003
Epoch 5/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 42s 456ms/step - accuracy: 0.6533 - loss: 0.9219 - val_accuracy: 0.4074 - val_loss: 1.4001
Epoch 6/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 42s 453ms/step - accuracy: 0.6744 - loss: 0.8696 - val_accuracy: 0.5000 - val_loss: 1.1513
Epoch 7/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 41s 449ms/step - accuracy: 0.6897 - loss: 0.8427 - val_accuracy: 0.5327 - val_loss: 1.1609
Epoch 8/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 41s 448ms/step - accuracy: 0.7156 - loss: 0.7942 - val_accu

In [10]:

# tuner = keras_tuner.RandomSearch(
#     build_model,
#     objective='val_loss',
#     max_trials=10)

In [11]:
# tuner.search(train_ds, epochs=5, validation_data=val_ds)
# best_model = tuner.get_best_models()[0]

In [12]:
# best_model.predict(val_ds)